# Rollout Perplexity Analysis

Can the model score its own outputs? Generate N rollouts (prefix=0), compute the
model's perplexity on each rollout and the ground-truth document, then check whether
lower perplexity correlates with higher contact accuracy.

In [ ]:
# Config
CHECKPOINT_PATH = "../../outputs/exp4"
PDB_ID = "1QYS"
CONTACT_DISTANCE_CUTOFF = 4.0
MAX_NEW_TOKENS = 3440
N_ROLLOUTS = 10
DEVICE = "cuda"

In [ ]:
# Download & parse PDB structure
import tempfile
import numpy as np
from biotite.database import rcsb
from biotite.structure.io import pdbx
from biotite.structure import filter_amino_acids

path = rcsb.fetch(PDB_ID, "cif", tempfile.gettempdir())
cif = pdbx.CIFFile.read(path)
atoms = pdbx.get_structure(cif.block, model=1)

first_chain = atoms.chain_id[0]
chain_atoms = atoms[(atoms.chain_id == first_chain) & filter_amino_acids(atoms) & (atoms.element != "H")]

res_ids = chain_atoms.res_id
unique_res = sorted(set(res_ids))
res_id_to_pos = {rid: i + 1 for i, rid in enumerate(unique_res)}
sequence = [chain_atoms[chain_atoms.res_id == rid].res_name[0] for rid in unique_res]
seq_len = len(sequence)
print(f"Protein {PDB_ID}: {seq_len} residues, chain {first_chain}")

In [ ]:
# Compute ground-truth contacts
from scipy.spatial import KDTree
from experiments.exp4_contact_prediction.src.data import VALID_ATOMS

coords = chain_atoms.coord
atom_names = chain_atoms.atom_name
atom_res_ids = chain_atoms.res_id

all_known_atoms = set()
for aa in VALID_ATOMS:
    all_known_atoms.update(VALID_ATOMS[aa])

tree = KDTree(coords)
pairs = tree.query_pairs(r=CONTACT_DISTANCE_CUTOFF)

gt_contacts = []
for i, j in pairs:
    ri, rj = atom_res_ids[i], atom_res_ids[j]
    pi, pj = res_id_to_pos.get(ri), res_id_to_pos.get(rj)
    if pi is None or pj is None:
        continue
    if abs(pi - pj) < 2:
        continue
    ai, aj = str(atom_names[i]), str(atom_names[j])
    if ai not in all_known_atoms or aj not in all_known_atoms:
        continue
    aa_i, aa_j = sequence[pi - 1], sequence[pj - 1]
    if aa_i not in VALID_ATOMS or ai not in VALID_ATOMS[aa_i]:
        continue
    if aa_j not in VALID_ATOMS or aj not in VALID_ATOMS[aa_j]:
        continue
    if pi > pj:
        gt_contacts.append((pj, pi, aj, ai))
    else:
        gt_contacts.append((pi, pj, ai, aj))

gt_contacts = sorted(set(gt_contacts), key=lambda c: (-abs(c[0] - c[1]), c[0], c[1], c[2], c[3]))
gt_pair_set = {(min(c[0], c[1]), max(c[0], c[1])) for c in gt_contacts}

seq_tokens = " ".join(f"<{aa}>" for aa in sequence)
base_prompt = f"<deterministic-positives-only> <begin_sequence> {seq_tokens} <begin_contacts>"
print(f"Ground-truth contacts: {len(gt_contacts)} ({len(gt_pair_set)} unique residue pairs)")

In [ ]:
# Load model & tokenizer
import math
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import LlamaForCausalLM
from experiments.exp4_contact_prediction.src.train import create_tokenizer, parse_generated_contacts

ckpt_base = Path(CHECKPOINT_PATH)
ckpt_dirs = sorted(ckpt_base.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
ckpt_path = ckpt_dirs[-1] if ckpt_dirs else ckpt_base
print(f"Loading checkpoint: {ckpt_path}")

tokenizer = create_tokenizer()
model = LlamaForCausalLM.from_pretrained(str(ckpt_path), torch_dtype=torch.bfloat16)
model = model.to(DEVICE).eval()
end_token_id = tokenizer.convert_tokens_to_ids("<end>")
begin_contacts_id = tokenizer.convert_tokens_to_ids("<begin_contacts>")
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Generate N rollouts (prefix=0)
import time

def run_generation(prompt, do_sample=False):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=do_sample,
            temperature=1.0 if do_sample else None,
            top_k=0 if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=end_token_id,
        )
    elapsed = time.time() - t0
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=False)
    contacts, valid = parse_generated_contacts(gen_text.split())
    return contacts, valid, elapsed

rollouts = []  # list of (contacts, valid_grammar)
for r in range(N_ROLLOUTS):
    contacts, valid, elapsed = run_generation(base_prompt, do_sample=True)
    rollouts.append((contacts, valid))
    print(f"Rollout {r:2d}: {len(contacts):4d} contacts, valid={valid}, {elapsed:.1f}s")

print(f"\nGenerated {N_ROLLOUTS} rollouts")

In [ ]:
# Compute perplexity for a document
def compute_perplexity(doc_text, region="contacts_only"):
    """Compute perplexity of a document.

    Args:
        doc_text: Full document as whitespace-separated tokens.
        region: 'full' for entire doc, 'contacts_only' for tokens after <begin_contacts>.

    Returns:
        (perplexity, n_tokens) - perplexity and number of tokens scored.
    """
    encoding = tokenizer(doc_text, return_tensors="pt", truncation=True, max_length=8192)
    input_ids = encoding["input_ids"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits

    # Shift: predict token t+1 from position t
    shift_logits = logits[0, :-1]  # (seq_len-1, vocab)
    shift_labels = input_ids[0, 1:]  # (seq_len-1,)
    per_token_loss = F.cross_entropy(shift_logits, shift_labels, reduction="none")

    if region == "contacts_only":
        # Find <begin_contacts> position
        ids = input_ids[0].tolist()
        bc_pos = None
        for idx, tid in enumerate(ids):
            if tid == begin_contacts_id:
                bc_pos = idx
                break
        if bc_pos is not None:
            # Loss indices are shifted by 1: loss[i] = loss of predicting token i+1
            # We want loss for predicting tokens after <begin_contacts>, so start at bc_pos
            loss_slice = per_token_loss[bc_pos:]
        else:
            loss_slice = per_token_loss
    else:
        loss_slice = per_token_loss

    avg_loss = loss_slice.mean().item()
    return math.exp(avg_loss), len(loss_slice)

print("Perplexity function ready.")

In [ ]:
# Build full documents and compute perplexity + accuracy for each rollout
def contacts_to_pair_set(contacts):
    return {(min(c[0], c[1]), max(c[0], c[1])) for c in contacts}

def build_document(contacts):
    """Build a full document string from sequence + contacts."""
    contact_toks = []
    for p1, p2, a1, a2 in contacts:
        contact_toks.extend([f"<p{p1}>", f"<p{p2}>", f"<{a1}>", f"<{a2}>"])
    return base_prompt + " " + " ".join(contact_toks) + " <end_contacts> <end>"

# Ground truth document
gt_doc = build_document(gt_contacts)
gt_ppl_contacts, gt_n_tok = compute_perplexity(gt_doc, region="contacts_only")
gt_ppl_full, gt_n_tok_full = compute_perplexity(gt_doc, region="full")
print(f"Ground truth: ppl_contacts={gt_ppl_contacts:.2f} ({gt_n_tok} tokens), "
      f"ppl_full={gt_ppl_full:.2f} ({gt_n_tok_full} tokens)")

# Each rollout
rollout_data = []  # list of dicts
for r, (contacts, valid) in enumerate(rollouts):
    doc = build_document(contacts)
    ppl_contacts, n_tok = compute_perplexity(doc, region="contacts_only")
    ppl_full, n_tok_full = compute_perplexity(doc, region="full")

    gen_pairs = contacts_to_pair_set(contacts)
    n_gen = len(gen_pairs)
    n_correct = len(gen_pairs & gt_pair_set)
    precision = n_correct / n_gen if n_gen > 0 else 0.0
    recall = n_correct / len(gt_pair_set) if gt_pair_set else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    rollout_data.append({
        "rollout": r,
        "n_contacts": len(contacts),
        "n_pairs": n_gen,
        "n_correct": n_correct,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "valid_grammar": valid,
        "ppl_contacts": ppl_contacts,
        "ppl_full": ppl_full,
        "n_tokens_contacts": n_tok,
        "n_tokens_full": n_tok_full,
    })
    print(f"Rollout {r:2d}: ppl_c={ppl_contacts:.2f} ppl_f={ppl_full:.2f} "
          f"| prec={precision:.1%} rec={recall:.1%} f1={f1:.3f} "
          f"| {n_gen} pairs, {n_correct} correct")

In [ ]:
# Summary table
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df = pd.DataFrame(rollout_data)
print(df[["rollout", "ppl_contacts", "ppl_full", "precision", "recall", "f1",
          "n_pairs", "n_correct", "n_contacts"]].to_string(index=False, float_format="%.3f"))

print(f"\nGround truth: ppl_contacts={gt_ppl_contacts:.2f}, ppl_full={gt_ppl_full:.2f}")
print(f"Rollout ppl_contacts: mean={df['ppl_contacts'].mean():.2f}, "
      f"std={df['ppl_contacts'].std():.2f}, "
      f"min={df['ppl_contacts'].min():.2f}, max={df['ppl_contacts'].max():.2f}")

In [ ]:
# Plot 1: Perplexity distribution with ground truth reference
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Contact-only perplexity
ax = axes[0]
ax.barh(range(N_ROLLOUTS), df["ppl_contacts"], color="#5C9BD5", label="Rollouts")
ax.axvline(gt_ppl_contacts, color="red", linewidth=2, linestyle="--", label="Ground truth")
ax.set_yticks(range(N_ROLLOUTS))
ax.set_yticklabels([f"R{r}" for r in range(N_ROLLOUTS)])
ax.set_xlabel("Perplexity (contacts only)")
ax.set_title("Contact-Region Perplexity")
ax.legend(fontsize=8)
# Annotate with F1
for i, row in df.iterrows():
    ax.text(row["ppl_contacts"] + 0.05, i, f"F1={row['f1']:.2f}",
            va="center", fontsize=7, color="#333")

# Full-doc perplexity
ax = axes[1]
ax.barh(range(N_ROLLOUTS), df["ppl_full"], color="#7BC47F", label="Rollouts")
ax.axvline(gt_ppl_full, color="red", linewidth=2, linestyle="--", label="Ground truth")
ax.set_yticks(range(N_ROLLOUTS))
ax.set_yticklabels([f"R{r}" for r in range(N_ROLLOUTS)])
ax.set_xlabel("Perplexity (full document)")
ax.set_title("Full-Document Perplexity")
ax.legend(fontsize=8)
for i, row in df.iterrows():
    ax.text(row["ppl_full"] + 0.02, i, f"F1={row['f1']:.2f}",
            va="center", fontsize=7, color="#333")

fig.suptitle(f"Perplexity per Rollout: {PDB_ID} (prefix=0)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Perplexity vs accuracy scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, metric, metric_label in zip(axes, ["precision", "recall", "f1"],
                                     ["Precision", "Recall", "F1"]):
    ax.scatter(df["ppl_contacts"], df[metric], s=60, color="#5C9BD5", zorder=3,
              label="Rollouts")

    # Label each point with rollout number
    for _, row in df.iterrows():
        ax.annotate(f"R{int(row['rollout'])}",
                    (row["ppl_contacts"], row[metric]),
                    textcoords="offset points", xytext=(5, 3), fontsize=7)

    # Correlation
    if len(df) > 2:
        corr = df["ppl_contacts"].corr(df[metric])
        ax.set_title(f"{metric_label} vs Contact PPL (r={corr:.2f})")
    else:
        ax.set_title(f"{metric_label} vs Contact PPL")

    ax.set_xlabel("Perplexity (contacts only)")
    ax.set_ylabel(metric_label)
    ax.grid(True, alpha=0.2)

    # Fit line if enough points
    if len(df) > 2:
        z = np.polyfit(df["ppl_contacts"], df[metric], 1)
        p = np.poly1d(z)
        x_range = np.linspace(df["ppl_contacts"].min(), df["ppl_contacts"].max(), 50)
        ax.plot(x_range, p(x_range), "--", color="grey", alpha=0.5, linewidth=1)

fig.suptitle(f"Does Lower Perplexity = Better Contacts? {PDB_ID} (prefix=0)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Per-contact-position perplexity curves (GT vs best/worst rollout)
def compute_per_position_ppl(doc_text):
    """Compute per-contact-group (4-token) perplexity.

    Returns list of (position_index, perplexity) tuples.
    """
    encoding = tokenizer(doc_text, return_tensors="pt", truncation=True, max_length=8192)
    input_ids = encoding["input_ids"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids).logits

    shift_logits = logits[0, :-1]
    shift_labels = input_ids[0, 1:]
    per_token_loss = F.cross_entropy(shift_logits, shift_labels, reduction="none")

    ids = input_ids[0].tolist()
    bc_pos = None
    for idx, tid in enumerate(ids):
        if tid == begin_contacts_id:
            bc_pos = idx
            break
    if bc_pos is None:
        return []

    contact_start = bc_pos + 1
    # Each contact is 4 tokens; loss index is shifted by 1
    results = []
    pos = 0
    while True:
        loss_start = contact_start + pos * 4 - 1
        loss_end = loss_start + 4
        if loss_end > len(per_token_loss):
            break
        avg = per_token_loss[loss_start:loss_end].mean().item()
        results.append((pos, math.exp(avg)))
        pos += 1

    return results

# GT curve
gt_curve = compute_per_position_ppl(gt_doc)

# Best and worst rollout by F1
best_idx = df["f1"].idxmax()
worst_idx = df["f1"].idxmin()
best_doc = build_document(rollouts[best_idx][0])
worst_doc = build_document(rollouts[worst_idx][0])
best_curve = compute_per_position_ppl(best_doc)
worst_curve = compute_per_position_ppl(worst_doc)

fig, ax = plt.subplots(figsize=(12, 4))

if gt_curve:
    positions, ppls = zip(*gt_curve)
    ax.plot(positions, ppls, linewidth=0.8, color="red", label=f"Ground truth (F1=n/a)", alpha=0.8)
if best_curve:
    positions, ppls = zip(*best_curve)
    ax.plot(positions, ppls, linewidth=0.8, color="blue",
            label=f"Best rollout R{best_idx} (F1={df.loc[best_idx, 'f1']:.3f})", alpha=0.8)
if worst_curve:
    positions, ppls = zip(*worst_curve)
    ax.plot(positions, ppls, linewidth=0.8, color="orange",
            label=f"Worst rollout R{worst_idx} (F1={df.loc[worst_idx, 'f1']:.3f})", alpha=0.8)

ax.set_xlabel("Contact Position Index")
ax.set_ylabel("Perplexity (per 4-token group)")
ax.set_title(f"Per-Contact Perplexity: GT vs Best/Worst Rollout ({PDB_ID})")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Rank rollouts by perplexity, overlay accuracy
sorted_df = df.sort_values("ppl_contacts").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(10, 5))

x = range(len(sorted_df))
color_ppl = "#5C9BD5"
color_f1 = "#E57373"

# Perplexity bars
bars = ax1.bar(x, sorted_df["ppl_contacts"], color=color_ppl, alpha=0.7, label="PPL (contacts)")
ax1.axhline(gt_ppl_contacts, color="red", linewidth=1.5, linestyle="--",
            label=f"GT PPL={gt_ppl_contacts:.2f}")
ax1.set_xlabel("Rollout (sorted by increasing PPL)")
ax1.set_ylabel("Perplexity (contacts)", color=color_ppl)
ax1.tick_params(axis="y", labelcolor=color_ppl)
ax1.set_xticks(x)
ax1.set_xticklabels([f"R{int(r)}" for r in sorted_df["rollout"]])

# F1 line on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, sorted_df["f1"], "o-", color=color_f1, linewidth=2, markersize=6, label="F1")
ax2.set_ylabel("F1 Score", color=color_f1)
ax2.tick_params(axis="y", labelcolor=color_f1)
ax2.set_ylim(0, 1)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

ax1.set_title(f"Rollouts Ranked by Perplexity: {PDB_ID} (prefix=0)")
plt.tight_layout()
plt.show()

# Print correlation summary
corr_contacts = df["ppl_contacts"].corr(df["f1"])
corr_full = df["ppl_full"].corr(df["f1"])
print(f"\nCorrelation (PPL_contacts vs F1): r = {corr_contacts:.3f}")
print(f"Correlation (PPL_full vs F1):     r = {corr_full:.3f}")
print(f"\nInterpretation: {'Negative correlation suggests the model CAN score its outputs' if corr_contacts < -0.3 else 'Weak/no correlation - model perplexity may not reliably rank output quality'}")